In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load the pre-cleaned dataset
df = pd.read_csv('../data/processed/aha_cleaned.csv')
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")

# --- Prepare labels ---
for col in ["Region Convert", "Control Convert"]:
    df[col] = df[col].astype(str).str.strip()

service_labels = {
    1: "Gen Med & Surg", 2: "Rehab", 3: "Orthopedic", 4: "Children's",
    6: "Surgical", 7: "Psychiatric", 8: "Alcoh & Chem Dependency",
    9: "Intellectual Disab", 10: "Long-term Acute", 11: "Heart",
    12: "Cancer", 13: "Women's; ObGyn", 14: "Other Specialty", 15: "Chronic Disease",
}
df["Service Type"] = df["Service"].map(service_labels)
df["Hospital Type"] = df["Gen Med & Sug or Not"].map({1: "General", 0: "Specialized"})

# --- Frequency table function ---
def frequency_table(data, column):
    counts = data[column].value_counts(dropna=False)
    pct = counts / counts.sum() * 100
    table = pd.DataFrame({
        column: counts.index,
        "Frequency": counts.values,
        "Relative Frequency (%)": pct.round(2).values,
        "Cumulative Frequency": counts.cumsum().values,
        "Cumulative (%)": pct.cumsum().round(2).values,
    })
    total = pd.DataFrame([{column: "Total", "Frequency": counts.sum(),
                           "Relative Frequency (%)": 100.0}])
    return pd.concat([table, total], ignore_index=True)

# --- Build the tables ---
ownership_freq = frequency_table(df, "Control Convert")
region_freq = frequency_table(df, "Region Convert")
service_freq = frequency_table(df, "Service Type")
type_freq = frequency_table(df, "Hospital Type")
crosstab_counts = pd.crosstab(df["Region Convert"], df["Control Convert"],
                              margins=True, margins_name="Total")
crosstab_pct = (pd.crosstab(df["Region Convert"], df["Control Convert"],
                            normalize="index") * 100).round(1)

for title, table in [("Table 1: Hospital Ownership", ownership_freq),
                     ("Table 2: Geographic Region", region_freq),
                     ("Table 3: Service Type", service_freq),
                     ("Table 4: General vs Specialized", type_freq),
                     ("Table 5: Ownership by Region (counts)", crosstab_counts),
                     ("Table 6: Ownership by Region (row %)", crosstab_pct)]:
    print(title)
    display(table)

# --- Save to outputs/tables ---
os.makedirs("../outputs/tables", exist_ok=True)
ownership_freq.to_csv("../outputs/tables/task4_freq_ownership.csv", index=False)
region_freq.to_csv("../outputs/tables/task4_freq_region.csv", index=False)
service_freq.to_csv("../outputs/tables/task4_freq_service.csv", index=False)
type_freq.to_csv("../outputs/tables/task4_freq_general_vs_specialized.csv", index=False)
crosstab_counts.reset_index().to_csv("../outputs/tables/task4_crosstab_region_ownership_counts.csv", index=False)
crosstab_pct.reset_index().to_csv("../outputs/tables/task4_crosstab_region_ownership_pct.csv", index=False)
print("Saved all Task 4 tables to outputs/tables/")

Loaded 1300 rows and 21 columns.
Table 1: Hospital Ownership


,Control Convert,Frequency,Relative Frequency (%),Cumulative Frequency,Cumulative (%)
0,Other Nonprofit,668,51.38,668.0,51.38
1,Gov NonFed,248,19.08,916.0,70.46
2,InvestorOwned ForProfit,220,16.92,1136.0,87.38
3,NonGov NonProfit,131,10.08,1267.0,97.46
4,Gov Fed,33,2.54,1300.0,100.00
5,Total,1300,100.00,NaN,NaN


Table 2: Geographic Region


,Region Convert,Frequency,Relative Frequency (%),Cumulative Frequency,Cumulative (%)
0,Pacific,223,17.15,223.0,17.15
1,South Atlantic,187,14.38,410.0,31.54
2,West North Central,187,14.38,597.0,45.92
3,East North Central,163,12.54,760.0,58.46
4,Mountain,138,10.62,898.0,69.08
5,West South Central,125,9.62,1023.0,78.69
6,Mid Atlantic,98,7.54,1121.0,86.23
7,East South Central,95,7.31,1216.0,93.54
8,New England,84,6.46,1300.0,100.00
9,Total,1300,100.00,NaN,NaN


Table 3: Service Type


,Service Type,Frequency,Relative Frequency (%),Cumulative Frequency,Cumulative (%)
0,Gen Med & Surg,954,73.38,954.0,73.38
1,Psychiatric,111,8.54,1065.0,81.92
2,Rehab,72,5.54,1137.0,87.46
3,Long-term Acute,68,5.23,1205.0,92.69
4,Children's,46,3.54,1251.0,96.23
5,Other Specialty,12,0.92,1263.0,97.15
6,Surgical,8,0.62,1271.0,97.77
7,Cancer,6,0.46,1277.0,98.23
8,Orthopedic,6,0.46,1283.0,98.69
9,Heart,5,0.38,1288.0,99.08


... (3 outputs left out)

Control Convert,Gov Fed,Gov NonFed,InvestorOwned ForProfit,NonGov NonProfit,Other Nonprofit,Total
Region Convert,,,,,,
East North Central,4,22,30,21,86,163
East South Central,1,32,20,10,32,95
Mid Atlantic,6,15,5,8,64,98
Mountain,5,24,25,15,69,138
New England,2,6,10,3,63,84
Pacific,2,42,30,28,121,223
South Atlantic,7,37,26,9,108,187
West North Central,5,53,32,19,78,187
West South Central,1,17,42,18,47,125


Table 6: Ownership by Region (row %)


Control Convert,Gov Fed,Gov NonFed,InvestorOwned ForProfit,NonGov NonProfit,Other Nonprofit
Region Convert,,,,,
East North Central,2.5,13.5,18.4,12.9,52.8
East South Central,1.1,33.7,21.1,10.5,33.7
Mid Atlantic,6.1,15.3,5.1,8.2,65.3
Mountain,3.6,17.4,18.1,10.9,50.0
New England,2.4,7.1,11.9,3.6,75.0
Pacific,0.9,18.8,13.5,12.6,54.3
South Atlantic,3.7,19.8,13.9,4.8,57.8
West North Central,2.7,28.3,17.1,10.2,41.7
West South Central,0.8,13.6,33.6,14.4,37.6


Saved all Task 4 tables to outputs/tables/
